# NB10 — Discovery Queue: motor de aristas (Fase 3b)

Reflejo del **motor de descubrimiento** bajo el plan ratificado por S5
(`docs/plans/graph_phase3_plan.vsm`, `@status ratified`, fase 3b).

- **Diferencia clave** — fase-2 DERIVA aristas de campos explícitos
  (`_fase2`: `design('CONTRADICTION')`, `status`, `claim_id`); fase-3
  **INFiere** aristas que no están escritas en ninguna fuente, a partir
  de la estructura del grafo + tiempo + contenido.
- **Semi-automático** — el motor PROPONE; S5 ratifica arista por arista
  (D-1, como gaps). NO-NEW-TRUTH: nunca escribe verdad nueva; la cola
  es candidatos con score cerrado + motivo.
- **Métodos determinísticos (R10)** —
  `link_prediction` (AA/CN/PA sobre no-dirigida por kind),
  `triad_closure` (A→B→C, falta A→C en las MISMAS kinds dirigidas;
  `contradicts` excluido — simétrico por diseño),
  `time_respecting` (A supersedes B con ts(A) > ts(B) sin arista).
- **Label-propagation propia** — orden fijo por id de nodo, sin random
  (Louvain excluido por componente aleatoria).
- **Defaults documentados (S5 'adelante' 2026-10-11, q1-q4 abiertas)** —
  q1: `contradicts` queda simétrico (patrón ratified); q2: umbral
  provisorio 0.5; q3: backlog 699 unmatched = ruido de sampling por
  ahora; q4: orden 3b→3c→3d.

*Ejecutar desde la raíz del repo:* `python3 /tmp/opencode/run_notebook.py docs/papers/attractor_convergence/analysis/10_discovery_queue.ipynb`

In [1]:
import sys, json, hashlib
sys.path.insert(0, "packages/framework")
from epistemic_graph import build, discover
from epistemic_graph.discovery import write_queue, MIN_SCORE_PROVISIONAL

g = build()
m = g["meta"]
print("fingerprint :", m["fingerprint"][:16], "| cache_hit:", m["cache_hit"])
print("nodes:", m["nodes_total"], "| edges:", m["edges_total"])

fingerprint : 727554991161b4c2 | cache_hit: True
nodes: 2104 | edges: 4777


## ① Denominadores del descubrimiento

Cada método reporta **sus** emisiones y el umbral aplicado (q2: 0.5
provisorio). El total de candidatos = suma de los tres métodos.

In [3]:
q = discover(g, min_score=MIN_SCORE_PROVISIONAL)
print(json.dumps(q["denominators"], indent=2))
print("total_candidates:", q["meta"]["total_candidates"])

{
  "link_prediction": {
    "kinds": [
      "claim",
      "predicate"
    ],
    "min_score": 0.5,
    "emitted": 49
  },
  "motif_closure": {
    "kinds": [
      "child_of",
      "derived_from",
      "supersedes"
    ],
    "emitted": 20
  },
  "time_respecting": {
    "kind": "supersedes",
    "emitted": 0
  },
  "communities_consenso": {
    "nodes": 382,
    "communities": 66
  }
}
total_candidates: 69


## ② Candidatos por método y kind

Desglose: cuántos candidatos produce cada método, y a qué kind de
arista apuntan. `time_respecting` puede ser 0 (honest: no hay pares
temporales con ts suficiente para supersedes faltante).

In [5]:
from collections import Counter
methods = Counter(c["method"] for c in q["candidates"])
kinds = Counter(c["kind"] for c in q["candidates"])
print("by method:", dict(methods))
print("by kind  :", dict(kinds))

by method: {'triad_closure': 20, 'link_prediction': 49}
by kind  : {'child_of': 20, 'correlates_with': 49}


## ③ Top candidatos (orden R10)

Ordenados por `(-score, kind, u, v)` — determinista. Cada candidato
lleva `score` cerrado en [0,1] + `method` + `evidence` (cn/aa/pa o via).

In [7]:
print(f"{'kind':<18} {'u':<45} {'v':<45} {'score':>6} {'method':<16}")
for c in q["candidates"][:15]:
    print(f"{c['kind']:<18} {c['u'][:45]:<45} {c['v'][:45]:<45} "
          f"{c['score']:>6.3f} {c['method']:<16}")

kind               u                                             v                                              score method          
child_of           predicate:AND(ctx_has($ctx, content), OR(ctx_ predicate:ctx_contains($ctx, content, from '/  1.000 triad_closure   
child_of           predicate:AND(ctx_has($ctx, content), OR(ctx_ predicate:ctx_contains($ctx, content, require  1.000 triad_closure   
child_of           predicate:OR(AND(ctx_has($ctx, content), OR(c predicate:OR(ctx_contains($ctx, content, from  1.000 triad_closure   
child_of           predicate:OR(AND(ctx_has($ctx, content), OR(c predicate:ctx_has($ctx, content)               1.000 triad_closure   
child_of           predicate:OR(AND(ctx_has($ctx, content), ctx_ predicate:ctx_contains($ctx, content, <<<<<<<  1.000 triad_closure   
child_of           predicate:OR(AND(ctx_has($ctx, content), ctx_ predicate:ctx_has($ctx, content)               1.000 triad_closure   
child_of           predicate:OR(AND(ctx_has($ctx, fileN

## ④ Comunidades determinísticas (label-prop)

Sobre la capa `consenso` (contradicts + supersedes). Número de
comunidades densas con pocas aristas entre ellas = frontera de
descubrimiento.

In [9]:
print("communities_consenso:", json.dumps(q["denominators"]["communities_consenso"], indent=2))

communities_consenso: {
  "nodes": 382,
  "communities": 66
}


## ⑤ R10 — mismo grafo → mismos candidatos

In [11]:
q1 = discover(g)
q2 = discover(g)
b1 = json.dumps(q1, sort_keys=True).encode()
b2 = json.dumps(q2, sort_keys=True).encode()
same = hashlib.sha256(b1).hexdigest() == hashlib.sha256(b2).hexdigest()
print("R10 determinism:", same, "| bytes:", len(b1))
assert same, "R10 VIOLATION"

R10 determinism: True | bytes: 24587


## ⑥ Snapshot + summary

Cola persistida en `state/epistemic/discovery_queue.jsonl` (append-only
convención; aquí un snapshot completo para inspección).

In [13]:
p = write_queue(q)
print("queue:", p, "| bytes:", p.stat().st_size)

summary = {
  "nb": "NB10 — discovery queue (Fase 3b, plan ratified S5 2026-10-11)",
  "plan": "docs/plans/graph_phase3_plan.vsm @status ratified",
  "graph": {
    "fingerprint": m["fingerprint"][:16],
    "nodes_total": m["nodes_total"],
    "edges_total": m["edges_total"],
  },
  "discovery": {
    "min_score": MIN_SCORE_PROVISIONAL,
    "total_candidates": q["meta"]["total_candidates"],
    "by_method": dict(Counter(c["method"] for c in q["candidates"])),
    "by_kind": dict(Counter(c["kind"] for c in q["candidates"])),
    "denominators": q["denominators"],
  },
  "defaults_documented": {
    "q1_contradicts": "simetrica (patron ratified fase-2); 3c decide",
    "q2_min_score": MIN_SCORE_PROVISIONAL,
    "q3_backlog_699": "ruido de sampling por ahora",
    "q4_order": "3b -> 3c -> 3d",
  },
  "checks": {
    "R10_determinism": "celda ⑤: same graph -> same candidates hash",
    "NO-NEW-TRUTH": "discover() pura; no muta el grafo",
    "semi_auto": "motor propone; S5 ratifica arista por arista",
    "queue": "state/epistemic/discovery_queue.jsonl (snapshot)",
  },
}
print(json.dumps(summary, indent=1, ensure_ascii=False))

queue: /home/rmw3/vOSlab/state/epistemic/discovery_queue.jsonl | bytes: 51533
{
 "nb": "NB10 — discovery queue (Fase 3b, plan ratified S5 2026-10-11)",
 "plan": "docs/plans/graph_phase3_plan.vsm @status ratified",
 "graph": {
  "fingerprint": "727554991161b4c2",
  "nodes_total": 2104,
  "edges_total": 4777
 },
 "discovery": {
  "min_score": 0.5,
  "total_candidates": 69,
  "by_method": {
   "triad_closure": 20,
   "link_prediction": 49
  },
  "by_kind": {
   "child_of": 20,
   "correlates_with": 49
  },
  "denominators": {
   "link_prediction": {
    "kinds": [
     "claim",
     "predicate"
    ],
    "min_score": 0.5,
    "emitted": 49
   },
   "motif_closure": {
    "kinds": [
     "child_of",
     "derived_from",
     "supersedes"
    ],
    "emitted": 20
   },
   "time_respecting": {
    "kind": "supersedes",
    "emitted": 0
   },
   "communities_consenso": {
    "nodes": 382,
    "communities": 66
   }
  }
 },
 "defaults_documented": {
  "q1_contradicts": "simetrica (patron rati